# 第三次作业-朴素贝叶斯源码分享

`sklearn.naive_bayes` 是一个纯 Python 文件（无 cpython ），5 个分类器继承同一套骨架。

In [1]:
import inspect
import warnings
import numpy as np
import sklearn
from scipy.special import logsumexp
from scipy.sparse import csr_matrix

from sklearn.naive_bayes import (
    GaussianNB, MultinomialNB, ComplementNB, BernoulliNB, CategoricalNB,
    _BaseNB, _BaseDiscreteNB,
)
from sklearn import naive_bayes as nb_module

print("sklearn :", sklearn.__version__)
print("source  :", inspect.getsourcefile(nb_module))
print("公开分类器:", [n for n in dir(nb_module) if n.endswith("NB") and not n.startswith("_")])

sklearn : 1.3.2
source  : /root/MSA/ENTER/envs/vmunet/lib/python3.8/site-packages/sklearn/naive_bayes.py
公开分类器: ['BernoulliNB', 'CategoricalNB', 'ComplementNB', 'GaussianNB', 'MultinomialNB']


## 2. 骨架：三级继承 + 模板方法

`naive_bayes.py` 的继承结构：

```
_BaseNB(ABC)                      —— 抽象 _joint_log_likelihood / _check_X
├── GaussianNB                    —— 连续型，直接挂 _BaseNB
└── _BaseDiscreteNB(ABC)          —— 再加抽象 _count / _update_feature_log_prob
    ├── MultinomialNB
    ├── ComplementNB
    ├── BernoulliNB
    └── CategoricalNB
```

关键设计：**基类负责所有公开 API，子类只填两个钩子函数**。
- `_BaseNB` 实现 `predict / predict_proba / predict_log_proba / predict_joint_log_proba`；
- `_BaseDiscreteNB` 把「计数 → 平滑 → 先验」的流水线只写一遍。
- 整族只要实现 _joint_log_likelihood (+ _check_X) 就能跑；离散族再多实现 _count / _update_feature_log_prob。

于是 5 个分类器的差别被压缩到极少数方法里。先看继承关系与抽象方法。

In [2]:
for cls in [GaussianNB, MultinomialNB, ComplementNB, BernoulliNB, CategoricalNB]:
    mro = " -> ".join(c.__name__ for c in cls.__mro__)
    print(f"{cls.__name__:16s}: {mro}")

print()
print("_BaseNB 的抽象方法       :", sorted(_BaseNB.__abstractmethods__))
print("_BaseDiscreteNB 的抽象方法:", sorted(_BaseDiscreteNB.__abstractmethods__))

GaussianNB      : GaussianNB -> _BaseNB -> ClassifierMixin -> BaseEstimator -> _MetadataRequester -> object
MultinomialNB   : MultinomialNB -> _BaseDiscreteNB -> _BaseNB -> ClassifierMixin -> BaseEstimator -> _MetadataRequester -> object
ComplementNB    : ComplementNB -> _BaseDiscreteNB -> _BaseNB -> ClassifierMixin -> BaseEstimator -> _MetadataRequester -> object
BernoulliNB     : BernoulliNB -> _BaseDiscreteNB -> _BaseNB -> ClassifierMixin -> BaseEstimator -> _MetadataRequester -> object
CategoricalNB   : CategoricalNB -> _BaseDiscreteNB -> _BaseNB -> ClassifierMixin -> BaseEstimator -> _MetadataRequester -> object

_BaseNB 的抽象方法       : ['_check_X', '_joint_log_likelihood']
_BaseDiscreteNB 的抽象方法: ['_count', '_joint_log_likelihood', '_update_feature_log_prob']


## 3. `_joint_log_likelihood` 的四个公开方法

四个公开方法的实现都只有几行，差别仅在是否归一化 / 在哪个域归一化：

- `predict` = `classes_[argmax(jll)]` —— **不做归一化**，因为 argmax 与常数无关；
- `predict_log_proba` = `jll − logsumexp(jll, axis=1)` —— 在**对数域**归一化；
- `predict_proba` = `exp(predict_log_proba)`；
- `predict_joint_log_proba` 直接把未归一化的 `jll` 暴露出来。

先看源码。

In [3]:
for name in ["predict_joint_log_proba", "predict", "predict_log_proba", "predict_proba"]:
    print()
    print(f"_BaseNB.{name}")
    print(inspect.getsource(getattr(_BaseNB, name)))


_BaseNB.predict_joint_log_proba
    def predict_joint_log_proba(self, X):
        """Return joint log probability estimates for the test vector X.

        For each row x of X and class y, the joint log probability is given by
        ``log P(x, y) = log P(y) + log P(x|y),``
        where ``log P(y)`` is the class prior probability and ``log P(x|y)`` is
        the class-conditional probability.

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            The input samples.

        Returns
        -------
        C : ndarray of shape (n_samples, n_classes)
            Returns the joint log-probability of the samples for each class in
            the model. The columns correspond to the classes in sorted
            order, as they appear in the attribute :term:`classes_`.
        """
        check_is_fitted(self)
        X = self._check_X(X)
        return self._joint_log_likelihood(X)


_BaseNB.predict
    def predict(self, X):
        ""

`predict_log_proba` 里的归一分母是 `logsumexp`，值得单独说清。

**定义**：$\mathrm{logsumexp}(x)=\log\sum_i e^{x_i}$，就是「log 域里的求和」。直觉写法是两步 —— `np.log(np.sum(np.exp(x)))`。

**为什么不能直接两步算**：`exp` 稍微偏离 0 就会出事 —— 输入偏大上溢成 `inf`、偏小下溢成 `0`，前者 `log(inf)=inf`，后者 `log(0)=-inf`。而高维高斯的 `jll` 恰好是一两万的极小负数，必然踩坑。

**稳定化**：注意到对任意常数 $m$ 都有

$$\log\sum_i e^{x_i}=m+\log\sum_i e^{x_i-m}.$$

取 $m=\max_i x_i$ 后，每一项 $e^{x_i-m}\le 1$、且至少有一项 $=1$，于是既不上溢、求和后取 log 也不会撞到 0。scipy 的实现正是这个套路。


In [4]:
import scipy.special._logsumexp as _lse

# scipy 的实现核心：先减去最大值 a_max，算完再加回
lines = inspect.getsource(_lse.logsumexp).splitlines()
i = next(k for k, l in enumerate(lines) if l.strip().startswith("a_max = np.amax"))
end = next(k for k, l in enumerate(lines) if l.strip() == "out += a_max")
print("\n".join(lines[i:end + 1]))

# 手写稳定版：m = max(x)，减 m 不改变结果
def my_logsumexp(x, axis=None, keepdims=False):
    m = np.max(x, axis=axis, keepdims=True)
    return m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=keepdims))

def as_scalar(v):
    return float(np.ravel(v)[0])

small = np.array([-1000.0, -1001.0, -1002.0])   # exp 全部下溢
large = np.array([ 1000.0,  1001.0])            # exp 全部上溢
with np.errstate(all="ignore"):
    print("\n天真版  小值 :", np.log(np.sum(np.exp(small))), "  <- log(0) = -inf")
    print("天真版  大值 :", np.log(np.sum(np.exp(large))), "  <- log(inf) = inf")

print("稳定版  小值 :", as_scalar(my_logsumexp(small)), "== scipy:", np.isclose(as_scalar(my_logsumexp(small)), logsumexp(small)))
print("稳定版  大值 :", as_scalar(my_logsumexp(large)), "== scipy:", np.isclose(as_scalar(my_logsumexp(large)), logsumexp(large)))

# 2D + axis=1：正是 predict_log_proba 的调用方式
A = np.array([[-7003.587, -7041.046], [-7027.278, -7064.371]])
print("axis=1 手写 == scipy:", np.allclose(my_logsumexp(A, axis=1, keepdims=True).ravel(), logsumexp(A, axis=1)))


    a_max = np.amax(a, axis=axis, keepdims=True)

    if a_max.ndim > 0:
        a_max[~np.isfinite(a_max)] = 0
    elif not np.isfinite(a_max):
        a_max = 0

    if b is not None:
        b = np.asarray(b)
        tmp = b * np.exp(a - a_max)
    else:
        tmp = np.exp(a - a_max)

    # suppress warnings about log of zero
    with np.errstate(divide='ignore'):
        s = np.sum(tmp, axis=axis, keepdims=keepdims)
        if return_sign:
            sgn = np.sign(s)
            s *= sgn  # /= makes more sense but we need zero -> zero
        out = np.log(s)

    if not keepdims:
        a_max = np.squeeze(a_max, axis=axis)
    out += a_max

天真版  小值 : -inf   <- log(0) = -inf
天真版  大值 : inf   <- log(inf) = inf
稳定版  小值 : -999.5923940355556 == scipy: True
稳定版  大值 : 1001.3132616875182 == scipy: True
axis=1 手写 == scipy: True


把 `logsumexp` 放回 NB 的场景：`jll` 很负时（高维高斯正是如此），先 `exp(jll)` 再按行归一化会**下溢成 0**、得到 `0/0 = nan`。下面用 5000 维高斯举例。

In [5]:
rng = np.random.RandomState(0)

# 高维 + 单位方差  =>  jll 量级约 -0.5 * d * log(2*pi)
Xd = rng.randn(400, 5000)
yd = np.r_[np.zeros(200, dtype=int), np.ones(200, dtype=int)]
big = GaussianNB().fit(Xd, yd)

jll = big.predict_joint_log_proba(Xd[:2])
print("jll (前两行)         :", jll)

naive = np.exp(jll)
naive = naive / naive.sum(axis=1, keepdims=True)
print("先 exp 再归一化       :", naive, "  <- 下溢")

print("sklearn predict_proba :", big.predict_proba(Xd[:2]))

manual_lp = jll - logsumexp(jll, axis=1, keepdims=True)
print("log_proba 等价        :", np.allclose(manual_lp, big.predict_log_proba(Xd[:2])))
print("predict 只是 argmax   :", np.array_equal(big.predict(Xd[:2]), big.classes_[np.argmax(jll, axis=1)]))

jll (前两行)         : [[-7003.58721888 -7041.04606285]
 [-7027.27755265 -7064.37082878]]
先 exp 再归一化       : [[nan nan]
 [nan nan]]   <- 下溢
sklearn predict_proba : [[1.00000000e+00 5.39300425e-17]
 [1.00000000e+00 7.77311083e-17]]
log_proba 等价        : True
predict 只是 argmax   : True


/tmp/ipykernel_71355/3200229934.py:12: RuntimeWarning: invalid value encountered in divide
  naive = naive / naive.sum(axis=1, keepdims=True)


## 4. 离散族的流水线：计数 → 平滑 → 先验

`_BaseDiscreteNB` 把离散模型共用的三步只写一遍（`fit` 与 `partial_fit` 都调用）：

1. `_count(X, Y)`：累加 `class_count_` / `feature_count_`（**子类各自实现**）；
2. `_check_alpha()`（基类统一实现）→ `_update_feature_log_prob(alpha)`（子类各自实现）：平滑后算 `log P(x_i|y)`；
3. `_update_class_log_prior(class_prior)`：算 `log P(y)`（**基类统一实现**）。

`_update_class_log_prior` 还藏了两个细节：`fit_prior=False` 时先验退化为均匀分布的 log 值 `−log(n_classes)`；某类未被观测时 `log(0)` 的 RuntimeWarning 被显式静默。

In [6]:
print(inspect.getsource(_BaseDiscreteNB.fit))
print()
print(inspect.getsource(_BaseDiscreteNB._update_class_log_prior))

    @_fit_context(prefer_skip_nested_validation=True)
    def fit(self, X, y, sample_weight=None):
        """Fit Naive Bayes classifier according to X, y.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Training vectors, where `n_samples` is the number of samples and
            `n_features` is the number of features.

        y : array-like of shape (n_samples,)
            Target values.

        sample_weight : array-like of shape (n_samples,), default=None
            Weights applied to individual samples (1. for unweighted).

        Returns
        -------
        self : object
            Returns the instance itself.
        """
        X, y = self._check_X_y(X, y)
        _, n_features = X.shape

        labelbin = LabelBinarizer()
        Y = labelbin.fit_transform(y)
        self.classes_ = labelbin.classes_
        if Y.shape[1] == 1:
            if len(self.classes_) == 2:
                Y = np.co

In [7]:
rng = np.random.RandomState(7)
Xp = rng.randint(0, 5, size=(6, 4))
yp = np.array([0, 0, 1, 1, 2, 2])

m_learn = MultinomialNB().fit(Xp, yp)
m_unif = MultinomialNB(fit_prior=False).fit(Xp, yp)

print("学到先验 class_log_prior_ :", m_learn.class_log_prior_)
print("均匀先验 class_log_prior_ :", m_unif.class_log_prior_)
print("均匀应等于 -log(3)        :", -np.log(3), "=>", np.allclose(m_unif.class_log_prior_, -np.log(3)))

学到先验 class_log_prior_ : [-1.09861229 -1.09861229 -1.09861229]
均匀先验 class_log_prior_ : [-1.09861229 -1.09861229 -1.09861229]
均匀应等于 -log(3)        : -1.0986122886681098 => True


## 5. 权重折叠进 one-hot 标签矩阵

`sample_weight` 没有进入任何 `_count` 实现，而是在 `fit` / `partial_fit` 里被**提前折叠进标签矩阵**：

```python
Y = Y.astype(np.float64, copy=False)      # 为支持权重，把 int64 转 float64
sample_weight = np.atleast_2d(sample_weight)
Y *= sample_weight.T                      # (n_samples, n_classes) 逐样本乘权重
```

之后 `_count` 里 `feature_count_ += Y.T @ X`、`class_count_ += Y.sum(axis=0)` 自动得到**加权计数**，子类一行权重判断都不用写。

这解释了两处「看着奇怪」的写法：
- 为什么先把 `Y` 从 int64 转 float64；
- `CategoricalNB._count` 里为何要判断 `Y.dtype.type == np.int64`（只有无权重时才走 `np.bincount` 不带 weights 的路径）。

In [8]:
src = inspect.getsource(_BaseDiscreteNB.fit)
lines = src.splitlines()
hits = [i for i, l in enumerate(lines) if "sample_weight" in l or "astype(np.float64" in l]
lo, hi = max(min(hits) - 1, 0), min(max(hits) + 1, len(lines) - 1)
print("\n".join(lines[lo:hi + 1]))

    @_fit_context(prefer_skip_nested_validation=True)
    def fit(self, X, y, sample_weight=None):
        """Fit Naive Bayes classifier according to X, y.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            Training vectors, where `n_samples` is the number of samples and
            `n_features` is the number of features.

        y : array-like of shape (n_samples,)
            Target values.

        sample_weight : array-like of shape (n_samples,), default=None
            Weights applied to individual samples (1. for unweighted).

        Returns
        -------
        self : object
            Returns the instance itself.
        """
        X, y = self._check_X_y(X, y)
        _, n_features = X.shape

        labelbin = LabelBinarizer()
        Y = labelbin.fit_transform(y)
        self.classes_ = labelbin.classes_
        if Y.shape[1] == 1:
            if len(self.classes_) == 2:
                Y = np.co

In [9]:
rng = np.random.RandomState(8)
Xc = rng.randint(0, 4, size=(6, 5))
yc = np.array([0, 0, 0, 1, 1, 1])

# 用 sample_weight：后三个样本权重为 2
w = np.array([1, 1, 1, 2, 2, 2])
m_w = MultinomialNB(force_alpha=True).fit(Xc, yc, sample_weight=w)

# 等价做法：把后三个样本物理复制一份
X_dup = np.vstack([Xc, Xc[3:]])
y_dup = np.concatenate([yc, yc[3:]])
m_d = MultinomialNB(force_alpha=True).fit(X_dup, y_dup)

print("sample_weight 的 feature_count_:\n", m_w.feature_count_)
print("物理复制 的 feature_count_    :\n", m_d.feature_count_)
print("feature_count_ 一致:", np.allclose(m_w.feature_count_, m_d.feature_count_))
print("class_count_   一致:", np.allclose(m_w.class_count_,   m_d.class_count_))

# 单类的退化处理：标签二值化后只有一列时，Y 被置为全 1
m_one = MultinomialNB().fit(Xc, np.zeros(6, dtype=int))
print("\n单个类别时 classes_:", m_one.classes_, " class_count_:", m_one.class_count_)

sample_weight 的 feature_count_:
 [[ 5.  1.  5.  1.  5.]
 [14. 10. 16.  6.  8.]]
物理复制 的 feature_count_    :
 [[ 5.  1.  5.  1.  5.]
 [14. 10. 16.  6.  8.]]
feature_count_ 一致: True
class_count_   一致: True

单个类别时 classes_: [0]  class_count_: [6.]


## 6. GaussianNB 的在线合并 —— Chan-Golub-LeVeque

常见的方差公式是 $\frac{1}{n}\sum(x-\mu)^2$，一次性算完。为了支持 `partial_fit`（在线学习），sklearn 不保存任何历史样本，直接合并旧统计量 + 新一批：

$$
\mu_{total}=\frac{n_{new}\mu_{new}+n_{past}\mu_{past}}{n_{total}},\qquad
SSD_{total}=SSD_{old}+SSD_{new}+\frac{n_{new}\,n_{past}}{n_{total}}\left(\mu_{past}-\mu_{new}\right)^2
$$

第三项（两组均值之差的加权平方）是关键的修正项。函数还处理了三个边界：`n_past==0`（首次）、`X.shape[0]==0`、`sample_weight.sum()≈0`。

参考：Stanford CS tech report STAN-CS-79-773（Chan, Golub, LeVeque）。

In [10]:
print(inspect.getsource(GaussianNB._update_mean_variance))

    @staticmethod
    def _update_mean_variance(n_past, mu, var, X, sample_weight=None):
        """Compute online update of Gaussian mean and variance.

        Given starting sample count, mean, and variance, a new set of
        points X, and optionally sample weights, return the updated mean and
        variance. (NB - each dimension (column) in X is treated as independent
        -- you get variance, not covariance).

        Can take scalar mean and variance, or vector mean and variance to
        simultaneously update a number of independent Gaussians.

        See Stanford CS tech report STAN-CS-79-773 by Chan, Golub, and LeVeque:

        http://i.stanford.edu/pub/cstr/reports/cs/tr/79/773/CS-TR-79-773.pdf

        Parameters
        ----------
        n_past : int
            Number of samples represented in old mean and variance. If sample
            weights were given, this should contain the sum of sample
            weights represented in old mean and variance.

        

In [11]:
rng = np.random.RandomState(1)
Xg = rng.randn(300, 4)
yg = rng.randint(0, 3, size=300)
classes = np.unique(yg)

single = GaussianNB().fit(Xg, yg)

batch = GaussianNB()
for chunk in np.array_split(np.arange(300), 7):
    batch.partial_fit(Xg[chunk], yg[chunk], classes=classes)

print("theta_ 分批 == 一次性:", np.allclose(single.theta_, batch.theta_))
print("var_   分批 == 一次性:", np.allclose(single.var_,   batch.var_))
print("max|Δtheta_| =", np.abs(single.theta_ - batch.theta_).max())
print("max|Δvar_|   =", np.abs(single.var_ - batch.var_).max())

theta_ 分批 == 一次性: True
var_   分批 == 一次性: True
max|Δtheta_| = 1.3877787807814457e-16
max|Δvar_|   = 1.6842549577233967e-10


## 7. `epsilon_` 的「先减后加」

`GaussianNB` 的 `var_` 里存的**不是原始方差**，而是**已经加过平滑量**的方差：

```
epsilon_ = var_smoothing * var(X, axis=0).max()
...
非首次 partial_fit:  self.var_[:, :] -= self.epsilon_   # 先扣掉
    ... 用干净的方差做第 6 节的在线合并 ...
                     self.var_[:, :] += self.epsilon_   # 再加回
```

若把「已加 epsilon」的方差直接喂进第 6 节的合并公式，epsilon 会被 `n_past·var` 项放大，并随每次 `partial_fit` 越滚越大。
> 注意：`epsilon_` 每次 `partial_fit` 都按**当前批次**重算（`var_smoothing * var(X_batch).max()`），所以严格说是「扣掉本批次的平滑量」。对同分布的批量数据前后 epsilon 近似相等，先减后加就能让合并公式拿到「干净」的方差。

In [12]:
src = inspect.getsource(GaussianNB._partial_fit)
lines = src.splitlines()
hits = [i for i, l in enumerate(lines) if "epsilon_" in l]
print("`epsilon_` 出现的相对行号:", hits)
print()
for i in hits:
    lo, hi = max(i - 2, 0), min(i + 2, len(lines) - 1)
    print(f"--- 上下文 (相对行 {i}) ---")
    print("\n".join(lines[lo:hi + 1]))
    print()

`epsilon_` 出现的相对行号: [41, 75, 107]

--- 上下文 (相对行 41) ---
        # boost the variance by epsilon, a small fraction of the standard
        # deviation of the largest dimension.
        self.epsilon_ = self.var_smoothing * np.var(X, axis=0).max()

        if first_call:

--- 上下文 (相对行 75) ---
                raise ValueError(msg % (X.shape[1], self.theta_.shape[1]))
            # Put epsilon back in each time
            self.var_[:, :] -= self.epsilon_

        classes = self.classes_

--- 上下文 (相对行 107) ---
            self.class_count_[i] += N_i

        self.var_[:, :] += self.epsilon_

        # Update if only no priors is provided



In [13]:
rng = np.random.RandomState(2)
Xe = rng.randn(200, 1)
ye = np.zeros(200, dtype=int)
VS = 0.5                                    # 故意用大 smoothing，让 epsilon 可见
chunks = np.array_split(np.arange(200), 4)

# sklearn 真实结果：连续多次 partial_fit
m_multi = GaussianNB(var_smoothing=VS)
for c in chunks:
    m_multi.partial_fit(Xe[c], ye[c], classes=np.array([0]))
print("sklearn 多次 partial_fit 的 var_ :", m_multi.var_[0, 0])

# 逐批次重算 epsilon_，复现 sklearn 的「先减后加」
def simulate(with_correction):
    mu, var, n = 0.0, 0.0, 0
    for c in chunks:
        Xc_ = Xe[c]
        eps_k = VS * np.var(Xc_, axis=0).max()          # 每次调用都按当前批次重算
        if n == 0:
            mu, var = GaussianNB._update_mean_variance(0, 0.0, 0.0, Xc_)
        else:
            v_used = var if not with_correction else var - eps_k
            mu, var = GaussianNB._update_mean_variance(n, mu, v_used, Xc_)
        var = var + eps_k
        n += len(c)
    return float(np.ravel(var)[0])

print("带修正(复现 sklearn) :", simulate(True), " -> 与上面一致:", np.isclose(simulate(True), m_multi.var_[0, 0]))
print("不带修正(错误做法)   :", simulate(False), " <- 被 epsilon 逐次撑大")

sklearn 多次 partial_fit 的 var_ : 1.7411603094330825
带修正(复现 sklearn) : 1.7411603094330825  -> 与上面一致: True
不带修正(错误做法)   : 2.626471643400318  <- 被 epsilon 逐次撑大


## 8. BernoulliNB 的代数化简 —— 消掉 `(1-X)`

Bernoulli 模型的对数似然是

$$\log P(x\mid y)=\sum_i\big[x_i\log p_i+(1-x_i)\log(1-p_i)\big].$$

朴素写法要构造 `(1-X)` 这个矩阵；sklearn 用恒等式

$$\sum_i (1-x_i)\log(1-p_i)=\sum_i\log(1-p_i)-\sum_i x_i\log(1-p_i)$$

把它化简成**一次稀疏点积 + 一个常数项**：

```python
neg_prob = np.log(1 - np.exp(feature_log_prob_))
jll = X @ (feature_log_prob_ - neg_prob).T + class_log_prior_ + neg_prob.sum(axis=1)
```

好处：**不物化 `(1-X)`**，直接支持 CSR 稀疏输入。

In [14]:
print(inspect.getsource(BernoulliNB._joint_log_likelihood))

    def _joint_log_likelihood(self, X):
        """Calculate the posterior log probability of the samples X"""
        n_features = self.feature_log_prob_.shape[1]
        n_features_X = X.shape[1]

        if n_features_X != n_features:
            raise ValueError(
                "Expected input with %d features, got %d instead"
                % (n_features, n_features_X)
            )

        neg_prob = np.log(1 - np.exp(self.feature_log_prob_))
        # Compute  neg_prob · (1 - X).T  as  ∑neg_prob - X · neg_prob
        jll = safe_sparse_dot(X, (self.feature_log_prob_ - neg_prob).T)
        jll += self.class_log_prior_ + neg_prob.sum(axis=1)

        return jll



In [15]:
rng = np.random.RandomState(3)
Xb = (rng.rand(100, 8) > 0.5).astype(float)
yb = rng.randint(0, 2, size=100)

bnb = BernoulliNB(binarize=None).fit(Xb, yb)

# 手工按原始公式（含 (1-X)）
flp = bnb.feature_log_prob_                       # log P(x_i=1 | y)
neg = np.log(1 - np.exp(flp))                     # log P(x_i=0 | y)
manual = Xb @ flp.T + (1 - Xb) @ neg.T + bnb.class_log_prior_
sk = bnb.predict_joint_log_proba(Xb)
print("手工公式 == sklearn:", np.allclose(manual, sk))

# 稀疏输入：sklearn 不构造 (1-X)，结果与稠密一致
Xsp = csr_matrix(Xb)
sk_sp = BernoulliNB(binarize=None).fit(Xsp, yb).predict_joint_log_proba(Xsp)
print("稀疏 == 稠密:", np.allclose(sk_sp, sk))

手工公式 == sklearn: True
稀疏 == 稠密: True


## 9. ComplementNB 用一个负号把 argmin 折进 argmax

Complement NB（Rennie et al. 2003）为类别不平衡设计，算法上是**最小化「补集」的权重**。

问题：基类 `predict` 写死用 **argmax**。sklearn 的解法是**把权重取负**：

```python
# _BaseNB.predict uses argmax, but ComplementNB operates with argmin.
feature_log_prob_ = -logged     # 于是 argmax(-s) ≡ argmin(s)
```

默认 `norm=False`（Mahout/Weka 的口味）；开启 `norm=True` 可切换到 Complement NB 论文（Rennie et al. 2003, Table 9）的计算方式。

In [16]:
print(inspect.getsource(ComplementNB._update_feature_log_prob))

    def _update_feature_log_prob(self, alpha):
        """Apply smoothing to raw counts and compute the weights."""
        comp_count = self.feature_all_ + alpha - self.feature_count_
        logged = np.log(comp_count / comp_count.sum(axis=1, keepdims=True))
        # _BaseNB.predict uses argmax, but ComplementNB operates with argmin.
        if self.norm:
            summed = logged.sum(axis=1, keepdims=True)
            feature_log_prob = logged / summed
        else:
            feature_log_prob = -logged
        self.feature_log_prob_ = feature_log_prob



In [17]:
rng = np.random.RandomState(4)
# 类别不平衡的计数型数据
Xc = rng.randint(0, 5, size=(300, 20))
yc = np.concatenate([np.zeros(280, dtype=int), np.ones(20, dtype=int)])

cnb = ComplementNB().fit(Xc, yc)
print("feature_log_prob_ shape:", cnb.feature_log_prob_.shape, " (已取负)")
print("predict_joint_log_proba head:", cnb.predict_joint_log_proba(Xc[:2]))

# 手工按论文：最小化 X @ log(comp)
comp_count = cnb.feature_all_ + cnb.alpha - cnb.feature_count_
logged = np.log(comp_count / comp_count.sum(axis=1, keepdims=True))
argmin_paper = np.argmin(Xc[:20] @ logged.T, axis=1)
argmax_sk = np.argmax(cnb.predict_joint_log_proba(Xc[:20]), axis=1)
print("论文 argmin == sklearn argmax:", np.array_equal(argmin_paper, argmax_sk))

norm_cnb = ComplementNB(norm=True).fit(Xc, yc)
print("norm=False 预测 [:12]:", cnb.predict(Xc[:12]))
print("norm=True  预测 [:12]:", norm_cnb.predict(Xc[:12]))

feature_log_prob_ shape: (2, 20)  (已取负)
predict_joint_log_proba head: [[105.12893999 104.58141516]
 [ 90.06660053  89.6454828 ]]
论文 argmin == sklearn argmax: True
norm=False 预测 [:12]: [0 0 0 0 0 0 0 0 0 0 0 0]
norm=True  预测 [:12]: [0 0 0 0 1 0 0 1 0 0 0 1]


## 10. CategoricalNB 的异构结构与动态增列

前四个变体的 `feature_log_prob_` 都是规整的 `(n_classes, n_features)` 二维矩阵；
**只有 CategoricalNB 是 list-of-arrays**：每个特征一张 `(n_classes, n_categories_i)` 的表，且列数会随 `partial_fit` 见到的新类别**动态增长**。

关键实现：
- `category_count_ = [np.zeros((n_classes, 0)) for _ in range(n_features)]` —— 每特征一张表，初始 0 列；
- `_count` 用 `np.pad(..., [(0,0),(0,diff)], "constant")` 按新类别补零列；
- 计数用 `np.bincount(X_feature[mask], weights=weights)`（按类掩码 + 加权直方图）；
- `_joint_log_likelihood` 用花式索引 `feature_log_prob_[i][:, indices].T`。

注：`_check_alpha` 在 alpha 上叠了「逐特征配置 + 1e-10 下限 + `force_alpha` 弃用过渡」三层逻辑，而且 **docstring 写 `default=False`，签名默认值却已是 `"warn"`**。

In [18]:
print(inspect.getsource(CategoricalNB._init_counters))
print("-" * 78)
print(inspect.getsource(CategoricalNB._count))
print("-" * 78)
print(inspect.getsource(_BaseDiscreteNB._check_alpha))

    def _init_counters(self, n_classes, n_features):
        self.class_count_ = np.zeros(n_classes, dtype=np.float64)
        self.category_count_ = [np.zeros((n_classes, 0)) for _ in range(n_features)]

------------------------------------------------------------------------------
    def _count(self, X, Y):
        def _update_cat_count_dims(cat_count, highest_feature):
            diff = highest_feature + 1 - cat_count.shape[1]
            if diff > 0:
                # we append a column full of zeros for each new category
                return np.pad(cat_count, [(0, 0), (0, diff)], "constant")
            return cat_count

        def _update_cat_count(X_feature, Y, cat_count, n_classes):
            for j in range(n_classes):
                mask = Y[:, j].astype(bool)
                if Y.dtype.type == np.int64:
                    weights = None
                else:
                    weights = Y[mask, j]
                counts = np.bincount(X_feature[mask], weights=weights

In [19]:
Xtoy = np.array([[0, 1], [1, 0], [2, 1], [0, 0], [1, 1], [2, 0]])
ytoy = np.array([0, 1, 0, 1, 0, 1])
cat = CategoricalNB().fit(Xtoy, ytoy)
print("type(feature_log_prob_):", type(cat.feature_log_prob_).__name__)
print("每特征表格的形状      :", [a.shape for a in cat.feature_log_prob_])
print("category_count_ 形状  :", [a.shape for a in cat.category_count_])

# 动态增列
cat2 = CategoricalNB()
cat2.partial_fit(np.array([[0], [1], [0]]), np.array([0, 1, 0]), classes=np.array([0, 1]))
print("\n第二次 partial_fit 前:", [a.shape for a in cat2.category_count_])
cat2.partial_fit(np.array([[3], [4]]), np.array([1, 0]))   # 出现新类别 3、4
print("第二次 partial_fit 后:", [a.shape for a in cat2.category_count_], " <- np.pad 补零列")

type(feature_log_prob_): list
每特征表格的形状      : [(2, 3), (2, 2)]
category_count_ 形状  : [(2, 3), (2, 2)]

第二次 partial_fit 前: [(2, 2)]
第二次 partial_fit 后: [(2, 5)]  <- np.pad 补零列


In [20]:
rng = np.random.RandomState(9)
Xa = rng.randint(0, 3, size=(6, 5))
ya = np.array([0, 0, 0, 1, 1, 1])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    MultinomialNB(alpha=0).fit(Xa, ya)
print("alpha=0 触发的告警:")
for x in caught:
    print("  ", x.category.__name__, "|", str(x.message)[:70])

m_a2 = MultinomialNB(alpha=0, force_alpha=True).fit(Xa, ya)
print("force_alpha=True 保留 alpha:", m_a2.alpha)

alpha=0 触发的告警:
   FutureWarning | The default value for `force_alpha` will change to `True` in 1.4. To s
   UserWarning | alpha too small will result in numeric errors, setting alpha = 1.0e-10
force_alpha=True 保留 alpha: 0


/root/MSA/ENTER/envs/vmunet/lib/python3.8/site-packages/sklearn/naive_bayes.py:903: RuntimeWarning: divide by zero encountered in log
  self.feature_log_prob_ = np.log(smoothed_fc) - np.log(


## 小结

回头看，`naive_bayes.py` 的设计主线其实只有三条。

**主线一：一切都在 log 域完成。**
整族分类器只实现一个 `_joint_log_likelihood`，把「先验 + 似然连乘」变成「对数相加」。`predict` 直接 `argmax`，连归一化都省掉；`predict_log_proba` 用 `logsumexp` 在 log 域归一化。5000 维实验里，朴素的「先 exp 再归一化」直接下溢成 `nan`，而 sklearn 仍能给出 `[1.0, 5.4e-17]`。

**主线二：凡是能增量化（`partial_fit`）的地方，全部改写。**
- 方差不从零重算，而用 Chan-Golub-LeVeque 公式合并「旧统计量 + 新批次」，不保留任何历史样本；
- 每次合并前先扣掉方差平滑量、合并后再加回，避免 `epsilon` 被反复放大；
- 类别数事先未知时，用 list-of-arrays + `np.pad` 让每个特征的类别表动态生长。

**主线三：数学上的等价变换，直接兑现成工程收益。**

| 等价变换 | 换来的工程收益 |
|---|---|
| `Σ[x·log p + (1−x)·log(1−p)]` 化简成一次点积 + 常数项 | BernoulliNB 不物化 `(1-X)`，直接吃 CSR 稀疏矩阵 |
| 补集权重取负：`argmax(−s) ≡ argmin(s)` | ComplementNB 复用基类写死的 argmax |
| 把权重乘进 one-hot 标签矩阵 | 5 个 `_count` 实现都对 `sample_weight` 完全无感 |
| 公开方法只留「要不要归一化 / 在哪个域归一化」的差异 | 整族被压成两个钩子函数 |

**还有一条隐藏的工程习惯**：把「选哪个分支」交给超参，把「数值安全」交给框架。`_check_alpha` 就是典型——逐特征 alpha + `1e-10` 下限 + `force_alpha` 弃用过渡，甚至连 docstring 写的默认值和签名都对不上。

一句话总结：**朴素贝叶斯在课本里是一行公式，在 sklearn 里是一套「log 域 + 可增量 + 稀疏友好」的工程骨架。读它的价值不在公式本身，而在这些为了数值稳定、内存与代码复用所做的取舍。**